In [ ]:
from datasets import load_dataset
raw_datasets = load_dataset("ArmelRandy/kde4")
raw_datasets

data/train-00000-of-00001-e6e984806935f4(…):   0%|          | 0.00/5.30M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20058 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['en', 'fr'],
        num_rows: 20058
    })
})

In [ ]:
raw_datasets["train"][0]

{'en': 'The Babelfish plugin can be accessed in the & konqueror; menubar under Tools Translate Web Page. Select from the list that drops down the language to translate from and the language to translate to.',
 'fr': 'Vous pouvez accéder au module externe Babelfish depuis la barre de menus de & konqueror;, dans Outils Traduire la page web. Choisissez dans la liste déroulante la langue du texte à traduire et la langue dans laquelle vous souhaitez la traduire.'}

In [ ]:
split_datasets = raw_datasets["train"].train_test_split(train_size=0.9, seed=20)
split_datasets

DatasetDict({
    train: Dataset({
        features: ['en', 'fr'],
        num_rows: 18052
    })
    test: Dataset({
        features: ['en', 'fr'],
        num_rows: 2006
    })
})

In [ ]:
split_datasets["validation"] = split_datasets.pop("test")
split_datasets

DatasetDict({
    train: Dataset({
        features: ['en', 'fr'],
        num_rows: 18052
    })
    validation: Dataset({
        features: ['en', 'fr'],
        num_rows: 2006
    })
})

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="pt")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [ ]:
def preprocess_data(records, max_length=128):
  inputs = [data for data in records["en"]]
  targets = [data for data in records["fr"]]
  model_inputs = tokenizer(
    inputs, text_target=targets, max_length=max_length, truncation=True
  )
  return model_inputs
tokenized_datasets = split_datasets.map(
    preprocess_data,
    batched=True,
    remove_columns=split_datasets["train"].column_names,
)
tokenized_datasets

Map:   0%|          | 0/18052 [00:00<?, ? examples/s]

Map:   0%|          | 0/2006 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 18052
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2006
    })
})

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
!pip install sacrebleu evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.3 MB/s eta 0:00:00


In [ ]:
import evaluate

metric = evaluate.load("sacrebleu")

In [ ]:
import numpy as np
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # In case the model returns more than the prediction logits
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100s in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import Seq2SeqTrainingArguments
model_name = f"marian-finetuned-kde4-en-to-fr"
YOUR_HF_USERNAME = "lilith-aeva"
args = Seq2SeqTrainingArguments(
    model_name,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=True,
    hub_model_id=f"{YOUR_HF_USERNAME}/{model_name}",
)

In [ ]:
from transformers import Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.evaluate(max_length=128)

{'eval_loss': 1.1564886569976807,
 'eval_model_preparation_time': 0.0053,
 'eval_bleu': 41.741432565771774,
 'eval_runtime': 151.2472,
 'eval_samples_per_second': 13.263,
 'eval_steps_per_second': 0.212}

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Model Preparation Time,Bleu
1,1.008563,0.905386,0.005300,49.241732
2,0.889217,0.876661,0.005300,49.932586
3,0.840237,0.863582,0.005300,50.351741
4,0.804991,0.860638,0.005300,50.424351


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2260, training_loss=0.8744879866068342, metrics={'train_runtime': 1365.22, 'train_samples_per_second': 52.891, 'train_steps_per_second': 1.655, 'total_flos': 2343630196703232.0, 'train_loss': 0.8744879866068342, 'epoch': 4.0})

In [ ]:
trainer.evaluate(max_length=128)

{'eval_loss': 0.8606377840042114,
 'eval_model_preparation_time': 0.0053,
 'eval_bleu': 50.35462392020589,
 'eval_runtime': 140.7817,
 'eval_samples_per_second': 14.249,
 'eval_steps_per_second': 0.227,
 'epoch': 4.0}

In [ ]:
trainer.push_to_hub("Training hoàn tất!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-kde4-en-to-fr/source.spm: 100%|##########|  778kB /  778kB            

  ...n-to-fr/training_args.bin: 100%|##########| 5.39kB / 5.39kB            

  ...-kde4-en-to-fr/target.spm: 100%|##########|  802kB /  802kB            

  ...n-to-fr/model.safetensors:  29%|##9       |  160MB /  542MB            

CommitInfo(commit_url='https://huggingface.co/lilith-aeva/marian-finetuned-kde4-en-to-fr/commit/fdae24ea0c361058b76d0a9ded1196820faf103c', commit_message='Training hoàn tất!', commit_description='', oid='fdae24ea0c361058b76d0a9ded1196820faf103c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/lilith-aeva/marian-finetuned-kde4-en-to-fr', endpoint='https://huggingface.co', repo_type='model', repo_id='lilith-aeva/marian-finetuned-kde4-en-to-fr'), pr_revision=None, pr_num=None)